# Colab 1 · Inspect before you trust
**Module 1 · Quality and Correctness of AI-Generated Code** · Glaucia Melo dos Santos, Toronto Metropolitan University

12 solutions, 4 tasks, 3 solutions each. The solutions and tests were **written for this class** to have visible differences in quality. They are not research data and not output from a particular model.

You will: (1) run the tests, (2) run two quality tools, (3) decide which solution you would keep for each task. **Your decision must cite at least two kinds of evidence** (for example tests and warnings, or warnings and complexity).

No key and no account are needed beyond opening this notebook.

In [ ]:
# 1. Set up (about a minute)
!pip -q install pylint radon pandas
import os, re, json, subprocess, sys, textwrap
import pandas as pd
os.makedirs("work", exist_ok=True)
print("ready")

ready


## 2. Write the 12 solutions and the tests to disk

In [ ]:
SOLUTIONS = {
 "T1_A": "def count_words(path):\n    with open(path, encoding=\"utf-8\") as f:\n        return len(f.read().split())\n",
 "T1_B": "def count_words(path):\n    # walks the file character by character\n    count = 0\n    in_word = False\n    with open(path, encoding=\"utf-8\") as f:\n        text = f.read()\n    for ch in text:\n        if ch == \" \" or ch == \"\\n\" or ch == \"\\t\" or ch == \"\\r\":\n            if in_word:\n                count = count + 1\n                in_word = False\n        else:\n            if not in_word:\n                in_word = True\n    if in_word:\n        count = count + 1\n    return count\n",
 "T1_C": "def count_words(path):\n    with open(path, encoding=\"utf-8\") as f:\n        text = f.read()\n    return len(text.split(\" \"))\n",
 "T2_A": "from datetime import datetime\n\n\ndef parse_date(s):\n    for fmt in (\"%Y-%m-%d\", \"%d/%m/%Y\"):\n        try:\n            d = datetime.strptime(s, fmt)\n            return (d.year, d.month, d.day)\n        except ValueError:\n            pass\n    raise ValueError(\"bad date: \" + s)\n",
 "T2_B": "def parse_date(s):\n    if \"-\" in s:\n        parts = s.split(\"-\")\n        if len(parts) == 3:\n            y = int(parts[0])\n            m = int(parts[1])\n            d = int(parts[2])\n            return (y, m, d)\n        else:\n            raise ValueError(\"bad date: \" + s)\n    elif \"/\" in s:\n        parts = s.split(\"/\")\n        if len(parts) == 3:\n            d = int(parts[0])\n            m = int(parts[1])\n            y = int(parts[2])\n            return (y, m, d)\n        else:\n            raise ValueError(\"bad date: \" + s)\n    else:\n        raise ValueError(\"bad date: \" + s)\n",
 "T2_C": "def parse_date(s):\n    if \"/\" in s:\n        a, b, c = s.split(\"/\")\n        return (int(c), int(a), int(b))\n    y, m, d = s.split(\"-\")\n    return (int(y), int(m), int(d))\n",
 "T3_A": "import heapq\n\n\ndef merge_sorted(a, b):\n    return list(heapq.merge(a, b))\n",
 "T3_B": "def merge_sorted(a, b):\n    out = []\n    while a and b:\n        if a[0] <= b[0]:\n            out.append(a.pop(0))\n        else:\n            out.append(b.pop(0))\n    out.extend(a)\n    out.extend(b)\n    return out\n",
 "T3_C": "def merge_sorted(a, b):\n    return sorted(a + b)\n",
 "T4_A": "import re\n\n_EMAIL = re.compile(r\"^[^@\\s]+@[^@\\s]+\\.[^@\\s]+$\")\n\n\ndef is_valid_email(s):\n    return bool(_EMAIL.match(s))\n",
 "T4_B": "def is_valid_email(s):\n    return \"@\" in s and \".\" in s\n",
 "T4_C": "import re\n\n_EMAIL = re.compile(r\"^[A-Za-z0-9]+(\\.[A-Za-z0-9]+)*@[A-Za-z0-9]+(\\.[A-Za-z]{2,3})$\")\n\n\ndef is_valid_email(s):\n    return bool(_EMAIL.match(s))\n"
}

TESTS = {
 "T1": "import os, importlib.util, pytest\nspec = importlib.util.spec_from_file_location(\"sol\", os.environ[\"SOL\"])\nsol = importlib.util.module_from_spec(spec); spec.loader.exec_module(sol)\n\n\ndef w(tmp_path, text):\n    p = tmp_path / \"f.txt\"; p.write_text(text, encoding=\"utf-8\"); return str(p)\n\n\ndef test_simple(tmp_path): assert sol.count_words(w(tmp_path, \"hello world\")) == 2\ndef test_newline_and_spaces(tmp_path): assert sol.count_words(w(tmp_path, \"a  b\\nc\")) == 3\ndef test_empty(tmp_path): assert sol.count_words(w(tmp_path, \"\")) == 0\ndef test_trailing_newline(tmp_path): assert sol.count_words(w(tmp_path, \"x y\\n\")) == 2\ndef test_tab(tmp_path): assert sol.count_words(w(tmp_path, \"a\\tb\")) == 2\n",
 "T2": "import os, importlib.util, pytest\nspec = importlib.util.spec_from_file_location(\"sol\", os.environ[\"SOL\"])\nsol = importlib.util.module_from_spec(spec); spec.loader.exec_module(sol)\n\n\ndef test_iso(): assert sol.parse_date(\"2026-11-13\") == (2026, 11, 13)\ndef test_slash(): assert sol.parse_date(\"13/11/2026\") == (2026, 11, 13)\ndef test_short_slash(): assert sol.parse_date(\"1/2/2026\") == (2026, 2, 1)\ndef test_leading_zero(): assert sol.parse_date(\"2026-02-03\") == (2026, 2, 3)\ndef test_invalid_month():\n    with pytest.raises(ValueError):\n        sol.parse_date(\"2026-13-45\")\n",
 "T3": "import os, importlib.util, pytest\nspec = importlib.util.spec_from_file_location(\"sol\", os.environ[\"SOL\"])\nsol = importlib.util.module_from_spec(spec); spec.loader.exec_module(sol)\n\n\ndef test_basic(): assert sol.merge_sorted([1, 3, 5], [2, 4, 6]) == [1, 2, 3, 4, 5, 6]\ndef test_empty_a(): assert sol.merge_sorted([], [1, 2]) == [1, 2]\ndef test_empty_b(): assert sol.merge_sorted([1, 2], []) == [1, 2]\ndef test_duplicates(): assert sol.merge_sorted([1, 2, 2], [2, 3]) == [1, 2, 2, 2, 3]\ndef test_inputs_unchanged():\n    a, b = [1, 3], [2, 4]\n    sol.merge_sorted(a, b)\n    assert a == [1, 3] and b == [2, 4]\n",
 "T4": "import os, importlib.util, pytest\nspec = importlib.util.spec_from_file_location(\"sol\", os.environ[\"SOL\"])\nsol = importlib.util.module_from_spec(spec); spec.loader.exec_module(sol)\n\n\ndef test_simple_valid(): assert sol.is_valid_email(\"a@b.com\")\ndef test_dotted_and_long_domain(): assert sol.is_valid_email(\"first.last@example.co.uk\")\ndef test_plus_tag(): assert sol.is_valid_email(\"a+tag@b.com\")\ndef test_no_at(): assert not sol.is_valid_email(\"plain\")\ndef test_no_dot_in_domain(): assert not sol.is_valid_email(\"a@b\")\ndef test_no_local_part(): assert not sol.is_valid_email(\"@b.com\")\ndef test_double_at(): assert not sol.is_valid_email(\"a@@b.com\")\ndef test_space(): assert not sol.is_valid_email(\"a b@c.com\")\n"
}

for name, src in SOLUTIONS.items():
    open(f"work/{name}.py", "w").write(src)
for task, src in TESTS.items():
    open(f"work/test_{task}.py", "w").write(src)
print(len(SOLUTIONS), "solutions,", len(TESTS), "test files written")

12 solutions, 4 test files written


## 3. Does it pass? Run the tests on all 12

**Pass rate = solutions that pass all tests ÷ solutions tried.** Always report the counts, for example "2 of 6", not only a percentage.

In [ ]:
def run_tests(name):
    task = name.split("_")[0]
    env = dict(os.environ, SOL=f"work/{name}.py")
    r = subprocess.run([sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider", f"work/test_{task}.py"], capture_output=True, text=True, env=env)
    last = r.stdout.strip().splitlines()[-1]
    passed = int((re.search(r"(\d+) passed", last) or [0, 0])[1])
    failed = int((re.search(r"(\d+) failed", last) or [0, 0])[1])
    return passed, passed + failed

rows = []
for name in SOLUTIONS:
    p, n = run_tests(name)
    rows.append({"solution": name, "tests_passed": p, "tests_total": n, "all_pass": p == n})
tests_df = pd.DataFrame(rows)
tests_df

,solution,tests_passed,tests_total,all_pass
0,T1_A,5,5,True
1,T1_B,5,5,True
2,T1_C,3,5,False
3,T2_A,5,5,True
4,T2_B,4,5,False
5,T2_C,2,5,False
6,T3_A,5,5,True
7,T3_B,4,5,False
8,T3_C,5,5,True
9,T4_A,8,8,True


In [ ]:
solutions_passing_all = int(tests_df.all_pass.sum())
print(f"{solutions_passing_all} of {len(tests_df)} solutions pass every test")
print("One solution = ", round(100/len(tests_df), 1), "points of pass rate")

6 of 12 solutions pass every test
One solution =  8.3 points of pass rate


## 4. Is it clean? Run the quality tools

* **pylint** counts warnings. Its message types are *convention, refactor, warning, error, fatal*. A style warning is not a bug: ask *what does this warning change in the behaviour?*
* **radon** gives a complexity grade. High complexity is a sign the code is harder to read and test, not an error.

In [ ]:
def lint(name):
    r = subprocess.run([sys.executable, "-m", "pylint", f"work/{name}.py", "--output-format=json", "--disable=C0114,C0115,C0116"], capture_output=True, text=True)
    msgs = json.loads(r.stdout or "[]")
    out = {t: 0 for t in ["fatal", "error", "warning", "refactor", "convention"]}
    for m in msgs:
        out[m["type"]] = out.get(m["type"], 0) + 1
    return out

from radon.complexity import cc_visit
def complexity(name):
    blocks = cc_visit(open(f"work/{name}.py").read())
    return max((b.complexity for b in blocks), default=0)

quality = []
for name in SOLUTIONS:
    q = lint(name); q["max_complexity"] = complexity(name); q["solution"] = name
    quality.append(q)
quality_df = pd.DataFrame(quality)[["solution", "fatal", "error", "warning", "refactor", "convention", "max_complexity"]]
quality_df

,solution,fatal,error,warning,refactor,convention,max_complexity
0,T1_A,0,0,0,0,1,1
1,T1_B,0,0,0,1,1,9
2,T1_C,0,0,0,0,1,1
3,T2_A,0,0,0,0,1,3
4,T2_B,0,0,0,2,1,5
5,T2_C,0,0,0,0,1,2
6,T3_A,0,0,0,0,1,1
7,T3_B,0,0,0,0,1,4
8,T3_C,0,0,0,0,1,1
9,T4_A,0,0,0,0,1,1


## 5. Your table: passed, warnings by severity, complexity

In [ ]:
table = tests_df.merge(quality_df, on="solution")
table["Passed (k of n)"] = table.tests_passed.astype(str) + " of " + table.tests_total.astype(str)
table = table[["solution", "Passed (k of n)", "error", "warning", "refactor", "convention", "max_complexity"]]
table["Decision (keep/drop)"] = ""
table["Evidence"] = ""
table   # copy this into your printed sheet, one row per solution (12 rows)

,solution,Passed (k of n),error,warning,refactor,convention,max_complexity,Decision (keep/drop),Evidence
0,T1_A,5 of 5,0,0,0,1,1,,
1,T1_B,5 of 5,0,0,1,1,9,,
2,T1_C,3 of 5,0,0,0,1,1,,
3,T2_A,5 of 5,0,0,0,1,3,,
4,T2_B,4 of 5,0,0,2,1,5,,
5,T2_C,2 of 5,0,0,0,1,2,,
6,T3_A,5 of 5,0,0,0,1,1,,
7,T3_B,4 of 5,0,0,0,1,4,,
8,T3_C,5 of 5,0,0,0,1,1,,
9,T4_A,8 of 8,0,0,0,1,1,,


## 6. Your decision

For each task choose the solution you would keep, and write the evidence in your sheet. A good decision:
1. cites **more than one** kind of evidence;
2. says **what is still missing** (what test or check would you ask for before shipping?).

Then open the sheet for the **evaluation protocol** and fill it as if you were publishing this result. The field most pairs leave blank is **Limits**.

### Sample-size reminder
With 12 solutions, one more or less moves the rate by 8.3 points. The study we discuss in class measures 15 solutions per approach, where one solution moves the rate by 6.7 points. Always say how many solutions your rate comes from.